In [3]:
import os

os.getcwd()

'/home/ec2-user/llm/singlegpu'

In [ ]:
import wandb

# API 키로 로그인
wandb.login(key="")

# 또는 대화형 로그인
wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ec2-user/.netrc


True

In [5]:
# wandb 사용 여부 설정
from datetime import datetime
use_wandb = True  # wandb를 사용하려면 True로 설정

# wandb 실행 이름 및 프로젝트 설정
wandb_project = "my-lora-project"
wandb_run_name = f"lora-llam3-8b-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"



# 토크나이저 및 데이터 준비

## Text Data 준비

In [6]:
from datasets import load_dataset
import pandas as pd
from huggingface_hub import login

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


/home/ec2-user/miniconda/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 허깅페이스 로그인 방법
my_hf_key=''
login(my_hf_key)

In [8]:
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")  

In [9]:
# 모델 레포지토리
model_path = "meta-llama/Llama-3.1-8B-Instruct"

# 데이터 path
data_path = "kyujinpy/KOR-OpenOrca-Platypus-v2"#"MarkrAI/KOpen-HQ-Hermes-2.5-60K"

In [10]:
# dataset 다운

data = load_dataset(data_path)

In [11]:
df = pd.DataFrame(data['train'])

In [12]:
# 데이터셋 구성확인
df.head()

,id,input,output,instruction
0,ko_platypus.0,,"모든 가능한 결과의 확률의 합이 1$이므로, 스피너가 $C$에 착륙할 확률을 구하려...","보드 게임 스피너는 $A$, $B$, $C$로 표시된 세 부분으로 나뉩니다. 스피너..."
1,ko_platypus.1,,14명 중 6명을 선택해야 하는데 순서는 중요하지 않습니다. 이것은 순열 문제가 아...,저희 학교 수학 클럽에는 남학생 6명과 여학생 8명이 있습니다. 주 수학 경시대회...
2,ko_platypus.2,,먼저 단어에 제한을 두지 않고 4글자로 된 모든 단어의 개수를 세어봅니다. 그런 다...,"자음이 하나 이상인 4글자 단어는 $A$, $B$, $C$, $D$, $E$로 몇 ..."
3,ko_platypus.3,,주사위 중 하나 이상이 1에 나올 때만 가능합니다. 두 주사위가 모두 1이 아닐 확...,멜린다는 표준 6면 주사위 두 개를 굴려서 굴린 두 개의 숫자로 두 자리 숫자를 만...
4,ko_platypus.4,,문제를 H와 T의 시퀀스라고 생각하세요. 두 개의 T가 연속으로 나타날 수 없으므로...,p$를 공정한 동전을 반복적으로 던지는 과정에서 5$의 앞면이 나오기 전에 2$의 ...


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForSeq2Seq
import bitsandbytes as bnb
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training)


In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [15]:
# 토크나이저 세팅: QLoRA시 pad 토큰을 eos로 설정해주기
bos = tokenizer.bos_token_id
eos = tokenizer.eos_token_id
pad = tokenizer.pad_token_id

tokenizer.pad_token_id = eos
tokenizer.padding_side = "right"
cut_off_len = 4098
val_size = 0.001
train_on_inputs = False
add_eos_token = False

In [16]:
template = {
    "prompt_input": "아래는 문제를 설명하는 지시사항과, 구체적인 답변을 방식을 요구하는 입력이 함께 있는 문장입니다. 이 요청에 대해 적절하게 답변해주세요.\n###입력:{input}\n###지시사항:{instruction}\n###답변:",
    "prompt_no_input": "아래는 문제를 설명하는 지시사항입니다. 이 요청에 대해 적절하게 답변해주세요.\n###지시사항:{instruction}\n###답변:"
}

In [17]:

from typing import Union

def generate_prompt(
    instruction: str,
    input: Union[None, str] = None,
    label: Union[None, str] = None,
    verbose: bool = False
) -> str:
    """
    주어진 instruction, input, label을 사용하여 프롬프트를 생성하는 함수.

    Parameters:
    - instruction (str): 문제 설명 또는 지시사항.
    - template (dict): 입력이 있는 경우와 없는 경우의 템플릿을 포함한 딕셔너리.
    - input (str or None): 문제에 대한 구체적인 입력 (옵션).
    - label (str or None): 정답 또는 응답 (옵션).
    - verbose (bool): 생성된 프롬프트를 출력할지 여부.

    Returns:
    - str: 완성된 프롬프트.
    """
    if input:
        res = template["prompt_input"].format(instruction=instruction, input=input)
    else:
        res = template["prompt_no_input"].format(instruction=instruction)

    if label:
        res = f"{res}{label}"

    if verbose:
        print(res)

    return res


In [18]:
def tokenize(prompt, add_eos_token=True):
   result = tokenizer(prompt,truncation=True,max_length=cut_off_len,padding=False,return_tensors=None,)
   if (result["input_ids"][-1] != tokenizer.eos_token_id
       and len(result["input_ids"]) < cut_off_len
       and add_eos_token
      ):
        result["input_ids"].append(tokenizer.eos_token_id)
        result["attention_mask"].append(1)

   result["labels"] = result["input_ids"].copy()
   return result

In [19]:
def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(
        data_point["instruction"],
        data_point["input"],
        data_point["output"]
        )
    tokenized_full_prompt = tokenize(full_prompt)
    if not train_on_inputs:
        user_prompt = generate_prompt(data_point["instruction"], data_point["input"])
        tokenized_user_prompt = tokenize(user_prompt, add_eos_token=add_eos_token)
        user_prompt_len = len(tokenized_user_prompt["input_ids"])

        if add_eos_token:
            user_prompt_len -= 1



        tokenized_full_prompt["labels"] = [-100] * user_prompt_len + tokenized_full_prompt["labels"][user_prompt_len:]

    return tokenized_full_prompt

In [20]:
if val_size > 0:
    train_val = data["train"].train_test_split(test_size=val_size, shuffle=True, seed=42)
    train_data = (train_val["train"].shuffle().map(
        generate_and_tokenize_prompt,
        cache_file_name="/data/train_cache"  # 캐시 파일 경로 지정
    ))
    val_data = (train_val["test"].shuffle().map(
        generate_and_tokenize_prompt,
        cache_file_name="/data/val_cache"  # 캐시 파일 경로 지정
    ).select(range(20)))
else:
    train_data = data["train"].shuffle().map(
        generate_and_tokenize_prompt,
        cache_file_name="/data/train_cache"  # 캐시 파일 경로 지정
    )
    val_data = None


In [21]:
val_data

Dataset({
    features: ['id', 'input', 'output', 'instruction', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 20
})

# Model 준비

In [22]:
# Quantization config 준비

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_storage=torch.bfloat16,
    )





In [23]:
# Model 로드 하기
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config = quantization_config,
    torch_dtype = torch.bfloat16,
    device_map = {"" : 0},
    cache_dir="/data/models"  # 모델이 다운로드되고 캐시될 경로

)


Loading checkpoint shards: 100%|██████████| 4/4 [00:58<00:00, 14.58s/it]


In [24]:
model = prepare_model_for_kbit_training(model)

In [25]:
config = LoraConfig(
    r = 16,
    lora_alpha = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
    )

In [26]:
def find_all_linear_names(model):
  cls = bnb.nn.Linear4bit
  lora_module_names = set()
  for name, module in model.named_modules():
    if isinstance(module, cls):
      names = name.split('.')
      lora_module_names.add(names[0] if len(names) == 1 else names[-1])
  return list(lora_module_names)

In [27]:
print('Trainable targer module:',find_all_linear_names(model))

Trainable targer module: ['k_proj', 'q_proj', 'v_proj', 'gate_proj', 'o_proj', 'up_proj', 'down_proj']


In [28]:
# QLoRA 준비
model = get_peft_model(model, config)

In [29]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [30]:
# 파라미터 수 체크
print_trainable_parameters(model)

trainable params: 13631488 || all params: 2809401344 || trainable%: 0.4852097059436731


In [ ]:
#Hyper parameter setting

output_dir='./llama_singleGPU-v1'

model_name = "llama3-lora"
num_epochs = 1
micro_batch_size = 1
gradient_accumulation_steps = 100
warmup_steps = 100
learning_rate = 5e-8
group_by_length = False
optimizer = 'paged_adamw_8bit'

# adam 활용시
beta1 = 0.9
beta2 = 0.95

lr_scheduler = 'cosine'
logging_steps = 1

use_wandb = True
wandb_run_name = 'Single_GPU_Optim'

use_fp16 = False
use_bf_16 = True
evaluation_strategy = 'steps'
eval_steps = 5
save_steps = 5
save_strategy = 'steps'




In [32]:
if use_wandb:
    wandb.init(
        project=wandb_project,
        name=wandb_run_name,
        config={
            "model": model_name,
            "learning_rate": learning_rate,
            "epochs": num_epochs,
            "batch_size": micro_batch_size,
            # 기타 추적하고 싶은 하이퍼파라미터
        }
    )


In [33]:
model.gradient_checkpointing_enable()

In [34]:
val_size

0.001

In [35]:
import numpy as np
import torch

# def compute_metrics(eval_preds):
#     logits, labels = eval_preds
    
#     # 패딩 토큰(-100) 마스킹
#     labels_flat = labels.flatten()
#     logits_flat = logits.reshape(-1, logits.shape[-1])
    
#     # -100 값 필터링
#     mask = labels_flat != -100
#     labels_filtered = labels_flat[mask]
#     logits_filtered = logits_flat[mask]
    
#     # 손실 계산
#     predictions = np.argmax(logits_filtered, axis=-1)
#     correct = predictions == labels_filtered
#     accuracy = correct.mean()
    
#     # 퍼플렉서티 계산
#     loss_fct = torch.nn.CrossEntropyLoss()
#     with torch.no_grad():
#         loss = loss_fct(torch.tensor(logits_filtered), torch.tensor(labels_filtered))
    
#     return {
#         "accuracy": float(accuracy),
#         "loss": float(loss)
#     }
# def compute_metrics(eval_preds):
#     with torch.no_grad():  # 그래디언트 계산 비활성화
#         logits, labels = eval_preds
        
#         # CPU로 이동
#         if isinstance(logits, torch.Tensor):
#             logits = logits.cpu().numpy()
#         if isinstance(labels, torch.Tensor):
#             labels = labels.cpu().numpy()
        
#         # 계산 수행
#         mask = labels.flatten() != -100
#         labels_filtered = labels.flatten()[mask]
#         predictions = np.argmax(logits.reshape(-1, logits.shape[-1])[mask], axis=-1)
#         accuracy = (predictions == labels_filtered).mean()
        
#         return {"accuracy": float(accuracy)}

In [ ]:
# from nltk.translate.bleu_score import sentence_bleu
# from rouge_score import rouge_scorer
# import torch
# import numpy as np
# from konlpy.tag import Okt

# # Initialize Korean tokenizer
# okt = Okt()

# def compute_metrics(eval_preds):
#     with torch.no_grad():
#         logits, labels = eval_preds
        
#         # CPU로 이동
#         if isinstance(logits, torch.Tensor):
#             logits = logits.cpu().numpy()
#         if isinstance(labels, torch.Tensor):
#             labels = labels.cpu().numpy()
        
#         # 토큰 ID를 텍스트로 변환
#         predictions = np.argmax(logits, axis=-1)
        
#         # 패딩 마스크 생성 (-100 값 처리)
#         pred_texts = []
#         label_texts = []
        
#         for pred, label in zip(predictions, labels):
#             # -100 값 제외하고 실제 토큰만 선택
#             label_mask = label != -100
#             true_label = label[label_mask]
            
#             # 예측 텍스트와 실제 텍스트 디코딩
#             pred_text = tokenizer.decode(pred[label_mask], skip_special_tokens=True)
#             label_text = tokenizer.decode(true_label, skip_special_tokens=True)
            
#             pred_texts.append(pred_text)
#             label_texts.append(label_text)
        
#         # ROUGE 점수 계산
#         scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
#         rouge_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
        
#         for pred, label in zip(pred_texts, label_texts):
#             scores = scorer.score(label, pred)
#             rouge_scores['rouge1'] += scores['rouge1'].fmeasure
#             rouge_scores['rouge2'] += scores['rouge2'].fmeasure
#             rouge_scores['rougeL'] += scores['rougeL'].fmeasure
        
#         # 평균 ROUGE 점수
#         num_samples = len(pred_texts)
#         rouge_scores = {k: v/num_samples for k, v in rouge_scores.items()}
        
#         # BLEU 점수 계산 - 한글 토큰화 적용
#         bleu_score = 0
#         for pred, label in zip(pred_texts, label_texts):
#             # KoNLPy의 Okt를 사용하여 한글 토큰화
#             reference = [okt.morphs(label)]
#             candidate = okt.morphs(pred)
#             bleu_score += sentence_bleu([reference[0]], candidate, weights=(0.25, 0.25, 0.25, 0.25))
        
#         bleu_score /= num_samples
        
#         # 결과 반환
#         results = {
#             'bleu': bleu_score,
#             'rouge1': rouge_scores['rouge1'],
#             'rouge2': rouge_scores['rouge2'],
#             'rougeL': rouge_scores['rougeL']
#         }
        
#         return results

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import torch
import numpy as np
from konlpy.tag import Okt

# Initialize Korean tokenizer
okt = Okt()

def compute_metrics(eval_preds):
    with torch.no_grad():
        logits, labels = eval_preds
        
        # CPU로 이동
        if isinstance(logits, torch.Tensor):
            logits = logits.cpu().numpy()
        if isinstance(labels, torch.Tensor):
            labels = labels.cpu().numpy()
        
        # 토큰 ID를 텍스트로 변환
        predictions = np.argmax(logits, axis=-1)
        
        # 패딩 마스크 생성 (-100 값 처리)
        pred_texts = []
        label_texts = []
        
        for pred, label in zip(predictions, labels):
            # -100 값 제외하고 실제 토큰만 선택
            label_mask = label != -100
            true_label = label[label_mask]
            
            # 예측 텍스트와 실제 텍스트 디코딩
            pred_text = tokenizer.decode(pred[label_mask], skip_special_tokens=True)
            label_text = tokenizer.decode(true_label, skip_special_tokens=True)
            
            pred_texts.append(pred_text)
            label_texts.append(label_text)
        
        # ROUGE 점수 계산
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        rouge_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
        
        for pred, label in zip(pred_texts, label_texts):
            scores = scorer.score(label, pred)
            rouge_scores['rouge1'] += scores['rouge1'].fmeasure
            rouge_scores['rouge2'] += scores['rouge2'].fmeasure
            rouge_scores['rougeL'] += scores['rougeL'].fmeasure
        
        # 평균 ROUGE 점수
        num_samples = len(pred_texts)
        rouge_scores = {k: v/num_samples for k, v in rouge_scores.items()}
        
        # BLEU 점수 계산 - 한글 토큰화 적용
        bleu_score = 0
        for pred, label in zip(pred_texts, label_texts):
            # KoNLPy의 Okt를 사용하여 한글 토큰화
            reference = [okt.morphs(label)]
            candidate = okt.morphs(pred)
            bleu_score += sentence_bleu([reference[0]], candidate, weights=(0.25, 0.25, 0.25, 0.25))
        
        bleu_score /= num_samples
        
        # 결과 반환
        results = {
            'bleu': bleu_score,
            'rouge1': rouge_scores['rouge1'],
            'rouge2': rouge_scores['rouge2'],
            'rougeL': rouge_scores['rougeL']
        }
        
        return results


In [37]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=micro_batch_size,
        per_device_eval_batch_size=micro_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=warmup_steps,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        adam_beta1=beta1,
        adam_beta2=beta2,
        fp16=use_fp16,
        bf16=use_bf_16,
        logging_steps=logging_steps,
        optim=optimizer,
        # 수정된 부분: evaluation_strategy → eval_strategy
        eval_strategy="steps",
        save_strategy="steps",
        eval_steps=eval_steps if val_size > 0 else None,
        save_steps=save_steps,
        lr_scheduler_type=lr_scheduler,
        load_best_model_at_end=True if val_size > 0 else False,
        group_by_length=group_by_length,
        report_to="wandb" if use_wandb else None,
        run_name=wandb_run_name if use_wandb else None,
    ),
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
    ),
    compute_metrics=compute_metrics

)



No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [38]:
# from transformers.trainer_callback import TrainerCallback
# import torch
# import numpy as np

# # 메모리 효율적인 평가를 위한 콜백
# class MemoryEfficientEvalCallback(TrainerCallback):
#     def on_evaluate(self, args, state, control, model, eval_dataloader, **kwargs):
#         # 평가 모드 설정
#         model.eval()
        
#         # 결과 저장 변수
#         total_correct = 0
#         total_samples = 0
        
#         # 배치 단위로 처리
#         with torch.no_grad():
#             for batch in eval_dataloader:
#                 # 데이터를 디바이스로 이동
#                 batch = {k: v.to(model.device) for k, v in batch.items()}
                
#                 # 모델 추론
#                 outputs = model(**batch)
#                 logits = outputs.logits
#                 labels = batch["labels"]
                
#                 # CPU로 이동하여 계산
#                 logits = logits.detach().cpu().numpy()
#                 labels = labels.detach().cpu().numpy()
                
#                 # 정확도 계산
#                 mask = labels.flatten() != -100
#                 filtered_labels = labels.flatten()[mask]
#                 filtered_logits = logits.reshape(-1, logits.shape[-1])[mask]
#                 predictions = np.argmax(filtered_logits, axis=-1)
                
#                 # 결과 누적
#                 total_correct += (predictions == filtered_labels).sum()
#                 total_samples += len(filtered_labels)
                
#                 # 메모리 정리
#                 del batch, outputs, logits, labels
#                 torch.cuda.empty_cache()
        
#         # 최종 정확도 계산
#         accuracy = total_correct / total_samples if total_samples > 0 else 0
        
#         # 결과 기록
#         state.log_history.append({
#             "eval_accuracy": float(accuracy),
#             "step": state.global_step
#         })
        
#         # 평가 완료 표시
#         control.should_evaluate = False
        
#         return control

# # 평가 배치 크기 최소화
# trainer.args.per_device_eval_batch_size = 1

# # 메모리 효율적인 평가 콜백 추가
# trainer.add_callback(MemoryEfficientEvalCallback())

# # compute_metrics 함수는 사용하지 않음
# trainer.compute_metrics = None


In [ ]:
model.config.use_cache = False


trainer.train()

Step,Training Loss,Validation Loss


In [ ]:
trainer.save_model()
tokenizer.save_pretrained(output_dir)
